# StockSense AI EXP-06 — gated Colab training
No training has been executed by creating this notebook. Choose a GPU runtime. Upload the reviewed source ZIP and an independently approved dataset; do not use the recovered EXP-05 snapshot as approved training data. The default configuration intentionally refuses training.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()  # Upload stocksense_exp06_source.zip produced locally
archive = Path("stocksense_exp06_source.zip")
assert archive.exists()
root = Path("/content/StockSense_EXP06")
root.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        target = (root / name).resolve()
        assert target.is_relative_to(root.resolve()), "Unsafe ZIP path"
    z.extractall(root)
%cd /content/StockSense_EXP06
import hashlib, json
manifest = json.loads((root / "source_manifest.json").read_text())
for name, expected in manifest.items():
    assert hashlib.sha256((root / name).read_bytes()).hexdigest() == expected, name


In [ ]:
%pip install -r research/EXP-06/requirements-colab.txt

In [ ]:
import sys, json, platform, hashlib, subprocess
import tensorflow as tf
import keras, numpy, pandas, sklearn
print({"python":sys.version,"tensorflow":tf.__version__,"keras":keras.__version__,"numpy":numpy.__version__,"pandas":pandas.__version__,"sklearn":sklearn.__version__})
print(tf.config.list_physical_devices("GPU"))
assert tf.config.list_physical_devices("GPU"), "Select Runtime > Change runtime type > GPU"
subprocess.run(["nvidia-smi"], check=False)

## Supply the approved inputs
Upload the approved CSV and completed experiment_config.json after provenance review. The config must contain the exact dataset hash, approval evidence and an untouched holdout boundary with enough sessions. Never turn the approval flags on merely to bypass an error.

In [ ]:
uploaded = files.upload()
dataset_path = Path("verified_RELIANCE_NS.csv")
config_path = Path("experiment_config.json")
assert dataset_path.exists() and config_path.exists()
config = json.loads(config_path.read_text())
actual_hash = hashlib.sha256(dataset_path.read_bytes()).hexdigest()
assert actual_hash == config["dataset_sha256"], "Dataset fingerprint mismatch"
print("Verified file hash:", actual_hash)

In [ ]:
from ml.research_v2.runner import run
output_dir = Path("research/EXP-06/artifacts/colab_run")
# Fails closed on an unapproved gate; final holdout is not scored here.
result = run(config, dataset_path, output_dir)
print(result)

In [ ]:
import shutil
archive = shutil.make_archive("/content/EXP06_colab_artifacts", "zip", output_dir)
files.download(archive)

Archive the exact input configuration and source ZIP alongside output artifacts. Review all fold results before selecting a candidate. Do not register a development fold checkpoint as a production model. Final holdout evaluation and prospective logging are later controlled stages.